# Semantic Search: Indexing and Retrieval
### Document: *1984* by George Orwell
This notebook applies the Load → Split → Embed → Retrieve pipeline on the classic novel *1984*.

## Setup — Install Dependencies

In [1]:
%pip install langchain-community langchain-huggingface langchain-text-splitters pypdf sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 25.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 332.2/332.2 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 1.6 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


## Step 1 — Load the Document

In [2]:
!curl -L -o 1984.pdf "https://www.planetebook.com/free-ebooks/1984.pdf"

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 1310k  100 1310k    0     0  10.0M      0 --:--:-- --:--:-- --:--:-- 10.0M


In [3]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("1984.pdf")
docs = loader.load()

print(f"Total pages loaded: {len(docs)}")
print(f"\n--- First 300 characters ---")
print(docs[0].page_content[:300])
print(f"\n--- Metadata ---")
print(docs[0].metadata)

Total pages loaded: 393

--- First 300 characters ---
Download free eBooks of classic literature, books and 
novels at Planet eBook. Subscribe to our free eBooks blog 
and email newsletter.
1984
By George Orwell

--- Metadata ---
{'producer': 'Adobe PDF Library 7.0', 'creator': 'Adobe InDesign CS2 (4.0)', 'creationdate': '2008-02-06T19:26:38+11:00', 'subject': 'Download classic literature as completely free eBooks from Planet eBook.', 'author': 'George Orwell', 'moddate': '2008-07-06T19:07:29+10:00', 'title': '1984', 'trapped': '/False', 'source': '1984.pdf', 'total_pages': 393, 'page': 0, 'page_label': '1'}


## Step 2 — Split into Chunks

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    add_start_index=True
)

all_splits = text_splitter.split_documents(docs)

print(f"Total chunks: {len(all_splits)}")
print(f"\n--- Example chunk ---")
print(all_splits[5].page_content)

Total chunks: 768

--- Example chunk ---
looked cold. Down in the street little eddies of wind were 
whirling dust and torn paper into spirals, and though the 
sun was shining and the sky a harsh blue, there seemed 
to be no colour in anything, except the posters that were 
plastered everywhere. The blackmoustachio’d face gazed 
down from every commanding corner. There was one on 
the house-front immediately opposite. BIG BROTHER IS 
WATCHING YOU, the caption said, while the dark eyes 
looked deep into Winston’s own. Down at street level an -
other poster, torn at one corner, flapped fitfully in the wind, 
alternately covering and uncovering the single word IN -
GSOC. In the far distance a helicopter skimmed down 
between the roofs, hovered for an instant like a bluebottle, 
and darted away again with a curving flight. It was the po -
lice patrol, snooping into people’s windows. The patrols did


## Step 3 — Embed and Store

In [5]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")

vector_store = InMemoryVectorStore(embeddings)
ids = vector_store.add_documents(documents=all_splits)

print(f"Stored {len(ids)} chunks in vector store!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Stored 768 chunks in vector store!


## Step 4 — Retrieve: Semantic Search

In [6]:
# Query 1: Big Brother
query1 = "Who is Big Brother and what does he represent?"
results1 = vector_store.similarity_search(query1, k=2)

for i, doc in enumerate(results1):
    print(f"--- Result {i+1} (Page {doc.metadata['page']}) ---")
    print(doc.page_content)
    print()

--- Result 1 (Page 261) ---
immediately below it. The consciousness of the masses 
needs only to be influenced in a negative way.
Given this background, one could infer, if one did not 
know it already, the general structure of Oceanic society. At 
the apex of the pyramid comes Big Brother. Big Brother is in-
fallible and all-powerful. Every success, every achievement, 
every victory, every scientific discovery, all knowledge, all 
wisdom, all happiness, all virtue, are held to issue directly 
from his leadership and inspiration. Nobody has ever seen 
Big Brother. He is a face on the hoardings, a voice on the 
telescreen. We may be reasonably sure that he will never die, 
and there is already considerable uncertainty as to when he 
was born. Big Brother is the guise in which the Party choos -
es to exhibit itself to the world. His function is to act as a 
focusing point for love, fear, and reverence, emotions which 
are more easily felt towards an individual than towards an

--- Result

In [7]:
# Query 2: Thought Police
query2 = "What is the Thought Police and how do they control people?"
results2 = vector_store.similarity_search(query2, k=2)

for i, doc in enumerate(results2):
    print(f"--- Result {i+1} (Page {doc.metadata['page']}) ---")
    print(doc.page_content)
    print()

--- Result 1 (Page 264) ---
dying, not only without any impulse to rebel, but without 
the power of grasping that the world could be other than it 
is. They could only become dangerous if the advance of in -
dustrial technique made it necessary to educate them more 
highly; but, since military and commercial rivalry are no 
longer important, the level of popular education is actually 
declining. What opinions the masses hold, or do not hold, 
is looked on as a matter of indifference. They can be granted 
intellectual liberty because they have no intellect. In a Party 
member, on the other hand, not even the smallest deviation 
of opinion on the most unimportant subject can be toler -
ated.
A Party member lives from birth to death under the eye 
of the Thought Police. Even when he is alone he can never be 
sure that he is alone. Wherever he may be, asleep or awake, 
working or resting, in his bath or in bed, he can be inspected

--- Result 2 (Page 151) ---
seriously of smashing your hea

In [8]:
# Query 3: Doublethink - with similarity scores
query3 = "What is doublethink?"
results3 = vector_store.similarity_search_with_score(query3, k=2)

for i, (doc, score) in enumerate(results3):
    print(f"--- Result {i+1} | Score: {score:.4f} | Page {doc.metadata['page']} ---")
    print(doc.page_content)
    print()

--- Result 1 | Score: 0.6892 | Page 269 ---
be carried out with sufficient precision, but it also has to be 
unconscious, or it would bring with it a feeling of falsity and 
hence of guilt. DOUBLETHINK lies at the very heart of In -
gsoc, since the essential act of the Party is to use conscious 
deception while retaining the firmness of purpose that goes 
with complete honesty. To tell deliberate lies while genu -
inely believing in them, to forget any fact that has become 
inconvenient, and then, when it becomes necessary again, 
to draw it back from oblivion for just so long as it is needed, 
to deny the existence of objective reality and all the while 
to take account of the reality which one denies—all this is 
indispensably necessary. Even in using the word DOUBLE -
THINK it is necessary to exercise DOUBLETHINK. For 
by using the word one admits that one is tampering with 
reality; by a fresh act of DOUBLETHINK one erases this 
knowledge; and so on indefinitely, with the lie alway

In [9]:
# Query 4: Using a Retriever with batch queries
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 2}
)

questions = ["What is Room 101?", "How does Winston feel about Julia?"]
batch_results = retriever.batch(questions)

for question, results in zip(questions, batch_results):
    print(f"=== {question} ===")
    for doc in results:
        print(f"  Page {doc.metadata['page']}: {doc.page_content[:200]}...")
    print()

=== What is Room 101? ===
  Page 356: move nothing, not even his head. A sort of pad gripped his 
head from behind, forcing him to look straight in front of 
him.
For a moment he was alone, then the door opened and 
O’Brien came in.
‘You ...
  Page 327: 19848
Winston lay silent. His breast rose and fell a little faster. 
He still had not asked the question that had come into his 
mind the first. He had got to ask it, and yet it was as though 
his t...

=== How does Winston feel about Julia? ===
  Page 217: der you to do so?’
‘Yes.’
‘You are prepared, the two of you, to separate and never 
see one another again?’
‘No!’ broke in Julia.
It appeared to Winston that a long time passed before 
he answered. Fo...
  Page 173: after their visit to the church belfry it had been impossible 
to arrange meetings. Working hours had been drastically 
increased in anticipation of Hate Week. It was more than 
a month distant, but t...

